# **Sesión 3:** Procesamiento de imagenes y extracción de características

## **Librerías**

In [1]:
import cv2
print("OpenCV should be 4.8.0.76 Current version:", cv2.__version__)
from typing import List
import numpy as np
import imageio
import copy
import glob
import os
import matplotlib.pyplot as plt
from typing import Optional
from utils import *

OpenCV should be 4.8.0.76 Current version: 4.8.0


In [2]:
def show_image(img):
    cv2.imshow("Image", img)
    cv2.waitKey()
    cv2.destroyAllWindows()
    
def write_image(img, filename="img.jpg"):
    cv2.imwrite(filename, img)

## **Apartado A:** Filtro Gaussiano y Detección de bordes: Sobel y Canny

El objetivo de este apartado es detectar los bordes de las imágenes de la carpeta ``data/partA-B``. Para ello, deberá seguir los siguientes pasos:

1. **Tarea A.1.** Defina el método ``gaussian_blur()`` que aplique un filtro gausiano para obtener imágenes borrosas. Siga todas las indicaciones del enunciado.
2. **Tarea A.2.** Aplique el método ``gaussian_blur()`` a todas las imágenes en ``data/partA-B``.


3. **Tarea A.3.** Defina la función ``sobel_edge_detector()`` que detecte bordes con el método Sobel. Siga todas las indicaciones del enunciado.
4. **Tarea A.4.** Aplique el método ``sobel_edge_detector()`` a todas las imágenes en ``data/partA-B``.


5. **Tarea A.5.** Defina la función ``canny_edge_detector()`` que detecte bordes con el método Canny. Siga todas las indicaciones del enunciado.
6. **Tarea A.6.** Aplique el método ``canny_edge_detector()`` a todas las imágenes en ``data/partA-B``.

### **Tarea A.1:** Defina el método ``gaussian_blur()`` que aplique un filtro gausiano para obtener imágenes borrosas.

In [3]:
# TODO Define the method
def gaussian_blur(img: np.array, sigma: float, filter_shape: Optional[List] = None, verbose: bool = False) -> np.array:
    # TODO If not given, compute the filter shape 
    if filter_shape == None:
        filter_l = 8*sigma
    else:
        filter_l = filter_shape[0]
    
    # TODO Create the filter coordinates matrices
    y, x = np.mgrid[0:filter_l, 0:filter_l]
    
    # TODO Define the formula that goberns the filter
    if sigma == 0:
        return None
    
    formula = lambda i,j: 1/(2*np.pi*sigma**2)*np.exp(-(i**2 +j**2)/(2*sigma**2))
    gaussian_filter = formula(x, y)
    
    # TODO Process the image
    gb_img = cv2.filter2D(img, -1, gaussian_filter)
    
    if verbose:
        show_image(img=gb_img, img_name=f"Gaussian Blur: Sigma = {sigma}")
    
    return gaussian_filter, gb_img.astype(np.uint8)

### **Tarea A.2.** Aplique el método ``gaussian_blur()`` a todas las imágenes en ``data``.

In [4]:
def load_images(filenames: List) -> List:
    return [cv2.imread(filename) for filename in filenames]

In [5]:
# TODO Get the gaussian blurred images using a list comprehension
gauss_sigma = 1
base_path = os.path.join(os.getcwd(), "../data/partA-B/*.jpg")

gb_path = glob.glob(base_path)
imgs = load_images(gb_path)
gb_imgs = [gaussian_blur(img, gauss_sigma) for img in imgs]

In [6]:
show_image(gb_imgs[2][1])

### **Tarea A.3:** Defina la función ``sobel_edge_detector()`` que detecte bordes con el método Sobel.

In [7]:
# TODO Define the method
def sobel_edge_detector(img: np.array, filter: np.array, gauss_sigma: float, gauss_filter_shape: Optional[List] = None, verbose: bool = False) -> np.array:
    # TODO Transform the img to grayscale
    gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # TODO Get a blurry img to improve edge detections
    blurred = gaussian_blur(img=gray_img, sigma=gauss_sigma, filter_shape=gauss_filter_shape, verbose=verbose)[1].astype(np.float32)
    
    # Re-scale
    blurred = blurred/255
    
    # TODO Get vertical edges
    v_edges = cv2.filter2D(gray_img, -1, filter)
    
    # TODO Transform the filter to get the orthogonal edges
    filter = filter.T
    
    # TODO Get horizontal edges
    h_edges = cv2.filter2D(gray_img, -1, filter)
    
    # TODO Get edges
    sobel_edges_img = np.hypot(h_edges, v_edges)
    sobel_edges_img = sobel_edges_img/sobel_edges_img.max()*255
    
    # TODO Get edges angle
    theta = np.arctan2(h_edges, v_edges)
    
    # Visualize if needed
    if verbose:
        show_image(img=sobel_edges_img, img_name="Sobel Edges")
    
    # TODO
    return np.squeeze(np.array(sobel_edges_img, dtype=np.uint8)), np.squeeze(theta)

### **Tarea A.4.** Aplique el método ``sobel_edge_detector()`` a todas las imágenes en ``data``.

In [8]:
# TODO Define a sigma value
gauss_sigma = 1

# TODO Define the Sobel filter
sobel_filter = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32)

# TODO Get the edges detected by Sobel using a list comprehension
sobel_edges_imgs = [sobel_edge_detector(img, sobel_filter, gauss_sigma) for img in imgs]

In [9]:
show_image(sobel_edges_imgs[0][0])

### **Tarea A.5:** Defina la función ``canny_edge_detector()`` que detecte bordes con el método Canny.

In [10]:
# TODO Define the method
def canny_edge_detector(img: np.array, sobel_filter: np.array, gauss_sigma: float, gauss_filter_shape: Optional[List] = None, verbose: bool = False):
    # TODO Call the method sobel_edge_detector()
    sobel_edges_img, theta = sobel_edge_detector(img, sobel_filter, gauss_sigma, gauss_filter_shape, verbose=verbose)
    
    # TODO Use NMS to refine edges
    canny_edges_img = non_max_suppression(sobel_edges_img, theta)
    
    if verbose:
        show_image(canny_edges_img, img_name="Canny Edges")
        
    return np.array(canny_edges_img, dtype=np.uint8) # Fixed output dtype

### **Tarea A.6.** Aplique el método ``canny_edge_detector()`` a todas las imágenes en ``data``.

In [11]:
# TODO Define Sobel filter
sobel_filter = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32)

# TODO Define a sigma value for Gauss
gauss_sigma = 5

# TODO Define a Gauss filter shape
gauss_filter_shape = [100, 100]

# TODO Get the edges detected by Canny using a list comprehension
canny_edges = [canny_edge_detector(img, sobel_filter, gauss_sigma, gauss_filter_shape) for img in imgs]

In [12]:
show_image(canny_edges[0])

### **Pregunta A.1:** Añada ruido a las imágenes de la carpeta ``data``. Compare los resultados que obtiene al aplicar su filtro Sobel con y sin filtro Gausiano.

In [13]:
def add_noise(img: np.array, mean: float, sigma: float):
    noise = np.array([[[np.random.normal(mean, sigma) for _ in range(img.shape[2])] for _ in range(img.shape[1])] for _ in range(img.shape[0])])
    
    noisy_image = img + noise
    
    noisy_image = np.clip(noisy_image, 0, 255)
    
    return np.array(noisy_image, dtype=np.uint8)

In [14]:
# TODO Homework
mean = 0 # 12
sigma = 17 # 81
base_path = os.path.join(os.getcwd(), "../data/partA-B/*.jpg")

gb_path = glob.glob(base_path)
imgs = load_images(gb_path)
noise_imgs = [add_noise(img, mean, sigma) for img in imgs]

In [15]:
# Sobel 

# Define a sigma value
gauss_sigma = 5

# Define the Sobel filter
sobel_filter = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float32)

# ===== NO NOISE =====
# Get the edges detected by Sobel using a list comprehension
sobel_edges_imgs = [sobel_edge_detector(img, sobel_filter, gauss_sigma) for img in imgs]

# ===== NOISE =====
sobel_edges_noisy_imgs = [sobel_edge_detector(img, sobel_filter, gauss_sigma) for img in noise_imgs]

In [16]:
show_image(sobel_edges_imgs[0][0])
show_image(sobel_edges_noisy_imgs[0][0])

### **Pregunta A.2:** Utilice la librería ``scikit-image`` y compare el efecto de los filtros Sobel, Canny y Prewitt sobre las imágenes de la carpeta ``data``. ¿Qué diferencias observa entre los filtros? ¿Puede obtener alguna conclusión y/o patrón?

In [17]:
import skimage as ski

In [18]:
# TODO Homework
gb_path = glob.glob(base_path)
imgs = load_images(gb_path)
gray_imgs = [ski.color.rgb2gray(img) for img in imgs]

sobel_scikit = [ski.filters.sobel(img) for img in gray_imgs]
canny_scikit = [ski.feature.canny(img).astype(np.uint8)*255 for img in gray_imgs]
prewitt_scikit = [ski.filters.prewitt(img) for img in gray_imgs]

# sobel_imgs = [sobel_edge_detector(img, sobel_filter, gauss_sigma) for img in imgs]
# canny_edges = [canny_edge_detector(img, sobel_filter, gauss_sigma, gauss_filter_shape) for img in imgs]

In [19]:
show_image(sobel_scikit[1])
show_image(canny_scikit[1])
show_image(prewitt_scikit[1])

## **Apartado B:** Operadores Morfológicos

Para resolver este partado, deberá seguir los siguientes pasos:

1. **Tarea B.1.** Defina el método ``binarize()`` para binarizar imágenes.
2. **Tarea B.2.** Defina el método ``custom_dilate()``.
3. **Tarea B.3.** Defina el método ``custom_erode()``.
4. **Pregunta B.1** Aplique los métodos ``custom_dilate()`` y ``custom_erode()`` a todas las imágenes de la carpeta ``data``.


### **Tarea B.1.** Defina el método ``binarize()`` para binarizar imágenes.

In [20]:
# TODO Homework: define the binarization method
def binarize(img: np.array, threshold: int = 127):
    binary_img = None
    return binary_img

### **Tarea B.2.** Defina el método ``custom_dilate()``

In [21]:
# TODO Homework: define the dilation method
def custom_dilate(img):
    # TODO pad the original image so it can keep dimensions after processing
    padded = np.pad()
    
    # TODO get img shape
    width = None
    height = None
    
    # TODO Create an element with the same dimensions as the padded img
    dilated = np.zeros()
    
    for j in range(height):
        for i in range(width):
            # TODO Add logic to the operation
            pass
            
    # TODO Select the region of interest (ROI). Modify if needed
    dilated = dilated[1:height+1, 1:width+1]
    
    return dilated

### **Tarea B.3.** Defina el método ``custom_erode()``

In [22]:
# TODO Homework: define the erotion method
def custom_erode(img):
    eroded = None
    
    return eroded

### **Pregunta B.1** Aplique los métodos ``custom_dilate()`` y ``custom_erode()`` a todas las imágenes de la carpeta ``data``.

In [23]:
# TODO Homework